# Chuyển Đổi Dữ Liệu Thành Unified Dataset
## Chi Tiết Từng File Dữ Liệu & Thống Kê

Notebook này:
- **Đọc** 24 file CSV từ `data/raw/`
- **Kiểm tra** chi tiết từng file (rows, columns, date range)
- **Tạo** unified dataset với 360 tháng (1995-2024)
- **Xử lý** missing values (2,633 → 0)
- **Xuất** chỉ 1 file đầu ra: `cpi_forecast_full_dataset_advanced.csv`

## Setup & Chuẩn Bị

In [40]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
script_dir = os.path.abspath('') 

data_raw_dir = os.path.join(script_dir, 'data', 'raw')
data_processed_dir = os.path.join(script_dir, 'data', 'processed')

os.makedirs(data_raw_dir, exist_ok=True)
os.makedirs(data_processed_dir, exist_ok=True)

print(f"Đang làm việc tại: {script_dir}")

Đang làm việc tại: d:\Nam4-HK1\KLTN\Vietnam_economic_lakehouse\model_test


## BƯỚC 1: Chi Tiết Từng File Dữ Liệu

In [41]:
data_config = {
    'cpi_mom.csv': {'date_col': 'date', 'value_cols': ['cpi', 'inflation'], 'category': 'Economic'},
    'core_inflation_rate.csv': {'date_col': 'date', 'value_cols': ['value'], 'category': 'Economic'},
    'cpi_base_year.csv': {'date_col': 'date', 'value_cols': ['base_2010'], 'category': 'Economic'},
    'interest_rate.csv': {'date_col': 'date', 'value_cols': ['interest_rate'], 'category': 'Economic'},
    'ppi_qoq.csv': {'date_col': 'date', 'value_cols': ['value'], 'category': 'Economic'},
    'm2.csv': {'date_col': 'date', 'value_cols': ['value'], 'category': 'Economic'},

    'brent.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'wti.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'gasoline.csv': {'date_col': 'date', 'value_cols': ['price'], 'category': 'Commodity'},
    'gasoline_world.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'natural_gas.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    
    'gold.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'silver.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    
    'VNINDEX.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    'VN30.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    'HNX.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    'UPCOM.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    
    'NASDAQ.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'S&P500.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'DAX.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'DOWJONES.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'NIKKEI225.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'HANGSENG.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    
    'USDVND.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Currency'},
}

print(f"{len(data_config)} files dữ liệu")

24 files dữ liệu


In [42]:
data_config = {
    # Economic
    'cpi_mom.csv': {'date_col': 'date', 'value_cols': ['cpi'], 'category': 'Economic'},
    'core_inflation_rate.csv': {'date_col': 'date', 'value_cols': ['value'], 'category': 'Economic'},
    'cpi_base_year.csv': {'date_col': 'date', 'value_cols': ['base_2010'], 'category': 'Economic'},
    'interest_rate.csv': {'date_col': 'date', 'value_cols': ['interest_rate'], 'category': 'Economic', 'filter_term': '3 Months'},
    'ppi_qoq.csv': {'date_col': 'date', 'value_cols': ['value'], 'category': 'Economic'},
    'm2.csv': {'date_col': 'date', 'value_cols': ['value'], 'category': 'Economic'},

    # Commodities
    'brent.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'wti.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'gasoline.csv': {'date_col': 'date', 'value_cols': ['price'], 'category': 'Commodity', 'filter_product': 'RON 95'},
    'gasoline_world.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'natural_gas.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'gold.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    'silver.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Commodity'},
    
    # Vietnam stocks
    'VNINDEX.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    'VN30.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    'HNX.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    'UPCOM.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'VN_Stock'},
    
    # Global stocks
    'NASDAQ.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'S&P500.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'DAX.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'DOWJONES.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'NIKKEI225.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    'HANGSENG.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Global_Stock'},
    
    # Currency
    'USDVND.csv': {'date_col': 'date', 'value_cols': ['close'], 'category': 'Currency'},
}

## BƯỚC 2: Tải & Kiểm Tra Chi Tiết

In [43]:
loaded_data = {}
file_details = []
failed = []

for filename in sorted(os.listdir(data_raw_dir)):
    if not filename.endswith('.csv'):
        continue
    
    try:
        filepath = os.path.join(data_raw_dir, filename)
        df = pd.read_csv(filepath)
        file_size = os.path.getsize(filepath) / 1024
        print(f"FILE: {filename}")
        print(f"Total rows: {len(df)}")
        print(f"Columns ({len(df.columns)}): {list(df.columns)}")
        # tỷ lệ null
        print("Null values (%):")
        print((df.isnull().mean() * 100).round(2))
        print(df.head())

        date_col = None
        if 'date' in df.columns:
            date_col = 'date'
        else:
            for col in df.columns:
                try:
                    test = pd.to_datetime(df[col].iloc[:10], errors='coerce')
                    if test.notna().sum() > 5:
                        date_col = col
                        print(f"Auto-detected date column: '{date_col}'")
                        break
                except:
                    continue
        
        if date_col is None:
            failed.append((filename, "No date column detected"))
            continue

        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        
        value_cols = [col for col in df.columns if col != date_col]
        print(f"Value columns to use ({len(value_cols)}): {value_cols}")

    except Exception as e:
        failed.append((filename, str(e)))
        print(f"ERROR: {str(e)[:80]}")

FILE: DAX.csv
Total rows: 9683
Columns (13): ['date', 'symbol', 'asset_class', 'unit', 'open', 'high', 'low', 'close', 'volume', 'change_percent', 'prev_close', 'change', 'source']
Null values (%):
date                0.0
symbol              0.0
asset_class         0.0
unit                0.0
open                0.0
high                0.0
low                 0.0
close               0.0
volume              0.0
change_percent      0.0
prev_close        100.0
change            100.0
source              0.0
dtype: float64
         date symbol asset_class unit         open         high          low  \
0  1987-12-30    DAX       index  EUR  1005.190002  1005.190002  1005.190002   
1  1988-01-04    DAX       index  EUR   956.489990   956.489990   956.489990   
2  1988-01-05    DAX       index  EUR   996.099976   996.099976   996.099976   
3  1988-01-06    DAX       index  EUR  1006.010010  1006.010010  1006.010010   
4  1988-01-07    DAX       index  EUR  1014.469971  1014.469971  1014.46997

In [44]:
loaded_data = {}
file_details = []
failed = []
null_analysis = []

for filename, config in data_config.items():
    try:
        filepath = os.path.join(data_raw_dir, filename)
        
        if not os.path.exists(filepath):
            print(f"{filename:25} | NOT FOUND")
            continue

        df = pd.read_csv(filepath)
        # Lấy 3M và RON 95 
        if filename == 'interest_rate.csv' and 'term' in df.columns:
            term_to_use = config.get('filter_term', '3M')
            df = df[df['term'] == term_to_use].copy()
        
        if filename == 'gasoline.csv' and 'product' in df.columns:
            product_to_use = config.get('filter_product', 'RON 95')
            df = df[df['product'] == product_to_use].copy()
        
        file_size = os.path.getsize(filepath) / 1024
        date_col = config['date_col']

        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

            value_cols = [col for col in config['value_cols'] if col in df.columns]
            
            if value_cols:
                subset = df[[date_col] + value_cols].copy()
                subset = subset.dropna(subset=[date_col] + value_cols)
                # chỉ xóa các cột có 100% null, còn lại giữ lại để phân tích
                for col in subset.columns:
                    if col != date_col:
                        null_count = subset[col].isnull().sum()
                        null_pct = (null_count / len(subset) * 100)
                        if null_pct > 0:
                            null_analysis.append({
                                'File': filename,
                                'Column': col,
                                'Null_Count': null_count,
                                'Null_Percent': f"{null_pct:.2f}%"
                            })
                subset = subset.dropna(axis=1, how='all')
                
                subset = subset.rename(columns={date_col: 'date'})
                for col in subset.columns:
                    if col != 'date':
                        if len(value_cols) == 1:
                            subset = subset.rename(columns={col: filename.replace('.csv', '')})
                        else:
                            subset = subset.rename(columns={col: f"{filename.replace('.csv', '')}_{col}"})
                
                loaded_data[filename] = subset

                file_info = {
                    'Filename': filename,
                    'Category': config['category'],
                    'Total_Rows': len(df),
                    'Data_Rows': len(subset),
                    'Date_Start': subset['date'].min().date(),
                    'Date_End': subset['date'].max().date(),
                    'Columns': len(subset.columns),
                }
                file_details.append(file_info)
                
                print(f"{filename:25} | {len(subset):6,} rows | {config['category']:15}")
    
    except Exception as e:
        failed.append((filename, str(e)))

cpi_mom.csv               |    360 rows | Economic       
core_inflation_rate.csv   |    131 rows | Economic       
cpi_base_year.csv         |     30 rows | Economic       
interest_rate.csv         |    109 rows | Economic       
ppi_qoq.csv               |     40 rows | Economic       
m2.csv                    |    195 rows | Economic       
brent.csv                 |  4,658 rows | Commodity      
wti.csv                   |  9,260 rows | Commodity      
gasoline.csv              |    125 rows | Commodity      
gasoline_world.csv        |  2,534 rows | Commodity      
natural_gas.csv           |  6,437 rows | Commodity      
gold.csv                  |  6,431 rows | Commodity      
silver.csv                |  6,433 rows | Commodity      
VNINDEX.csv               |  6,240 rows | VN_Stock       
VN30.csv                  |  4,290 rows | VN_Stock       
HNX.csv                   |  5,071 rows | VN_Stock       
UPCOM.csv                 |  4,177 rows | VN_Stock       
NASDAQ.csv    

### Chi Tiết Từng File

In [45]:
if null_analysis:
    null_df = pd.DataFrame(null_analysis)
    for file in null_df['File'].unique():
        file_nulls = null_df[null_df['File'] == file]
        print(f"{file}:")
        for _, row in file_nulls.iterrows():
            print(f"{row['Column']:30} | {row['Null_Count']:5} nulls ({row['Null_Percent']})")
else:
    print("No null value")

No null value


In [46]:
file_details_df = pd.DataFrame(file_details)

for category in file_details_df['Category'].unique():
    category_files = file_details_df[file_details_df['Category'] == category]
    total_rows = category_files['Data_Rows'].astype(int).sum()
    print(f"  • {category:20} | {len(category_files):2} files | {total_rows:,} total rows")

for idx, row in file_details_df.iterrows():
    print(f"""
  File {idx+1}: {row['Filename']}
    - Category: {row['Category']}
    - Rows: {row['Total_Rows']} (sau lọc: {row['Data_Rows']})
    - Date Range: {row['Date_Start']} to {row['Date_End']}
    - Columns: {row['Columns']}
  """)

  • Economic             |  6 files | 865 total rows
  • Commodity            |  7 files | 35,878 total rows
  • VN_Stock             |  4 files | 19,778 total rows
  • Global_Stock         |  6 files | 81,686 total rows
  • Currency             |  1 files | 8,264 total rows

  File 1: cpi_mom.csv
    - Category: Economic
    - Rows: 360 (sau lọc: 360)
    - Date Range: 1995-01-01 to 2024-12-01
    - Columns: 2
  

  File 2: core_inflation_rate.csv
    - Category: Economic
    - Rows: 131 (sau lọc: 131)
    - Date Range: 2015-04-01 to 2026-02-01
    - Columns: 2
  

  File 3: cpi_base_year.csv
    - Category: Economic
    - Rows: 30 (sau lọc: 30)
    - Date Range: 1995-01-01 to 2024-01-01
    - Columns: 2
  

  File 4: interest_rate.csv
    - Category: Economic
    - Rows: 109 (sau lọc: 109)
    - Date Range: 2014-04-01 to 2026-04-14
    - Columns: 2
  

  File 5: ppi_qoq.csv
    - Category: Economic
    - Rows: 40 (sau lọc: 40)
    - Date Range: 2016-01-01 to 2025-10-01
    - Columns:

## BƯỚC 3: Gộp Tần Suất Thành Hàng Tháng

In [47]:
min_date = pd.Timestamp('2018-06-01')
max_date = pd.Timestamp.now()
time_axis = pd.date_range(start=min_date, end=max_date, freq='MS')
time_index = pd.DataFrame({'date': time_axis})

print(f"  Standard time axis: {time_axis[0].date()} to {time_axis[-1].date()}")
print(f"  Total months: {len(time_axis)}")

  Standard time axis: 2018-06-01 to 2026-04-01
  Total months: 95


In [48]:
frequency_analysis = []

for filename, config in data_config.items():
    try:
        filepath = os.path.join(data_raw_dir, filename)
        
        if not os.path.exists(filepath):
            continue

        df = pd.read_csv(filepath)

        if filename == 'interest_rate.csv' and 'term' in df.columns:
            term_to_use = config.get('filter_term', '3M')
            df = df[df['term'] == term_to_use].copy()
        
        if filename == 'gasoline.csv' and 'product' in df.columns:
            product_to_use = config.get('filter_product', 'RON 95')
            df = df[df['product'] == product_to_use].copy()
        
        date_col = config['date_col']
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

            df_sorted = df.sort_values(date_col)
            dates = df_sorted[date_col].unique()
            
            if len(dates) > 1:
                date_diffs = pd.Series(dates[1:]) - pd.Series(dates[:-1])
                min_diff = date_diffs.min().days
                max_diff = date_diffs.max().days
                mode_diff = date_diffs.mode()[0].days if len(date_diffs.mode()) > 0 else None

                if mode_diff == 1:
                    freq_type = "DAILY"
                elif mode_diff == 7:
                    freq_type = "WEEKLY"
                elif 28 <= mode_diff <= 31:
                    freq_type = "MONTHLY"
                elif mode_diff == 365 or mode_diff == 366:
                    freq_type = "ANNUAL"
                else:
                    freq_type = f"OTHER ({mode_diff}d)"
                
                rows_per_day = len(df) / ((df[date_col].max() - df[date_col].min()).days + 1)
                
                frequency_analysis.append({
                    'File': filename,
                    'Category': config['category'],
                    'Total_Rows': len(df),
                    'Date_Range': f"{df_sorted[date_col].min().date()} to {df_sorted[date_col].max().date()}",
                    'Frequency': freq_type,
                    'Min_Gap_Days': min_diff,
                    'Max_Gap_Days': max_diff,
                    'Rows_Per_Day': f"{rows_per_day:.2f}",
                    'Needs_Resample': "NO" if freq_type == "MONTHLY" else "YES"
                })
    
    except Exception as e:
        pass

freq_df = pd.DataFrame(frequency_analysis)

for freq in freq_df['Frequency'].unique():
    files_with_freq = freq_df[freq_df['Frequency'] == freq]
    print(f"\n{freq:10} ({len(files_with_freq)} files):")
    for _, row in files_with_freq.iterrows():
        needs = "RESAMPLE" if row['Needs_Resample'] == "YES" else "KEEP AS-IS"
        print(f"  • {row['File']:25} | {row['Total_Rows']:6,} rows | {needs}")


print(f"{'File':<25} {'Category':<18} {'Frequency':<10} {'Total Rows':<12} {'Resample?':<12}")
print("-" * 120)

for _, row in freq_df.iterrows():
    needs = row['Needs_Resample']
    print(f"{row['File']:<25} {row['Category']:<18} {row['Frequency']:<10} {row['Total_Rows']:<12,} {needs:<12}")

resample_count = (freq_df['Needs_Resample'] == "YES").sum()
keep_count = (freq_df['Needs_Resample'] == "NO").sum()

print("\n" + "="*120)
print(f"KHÔNG RESAMPLE (đã monthly): {keep_count} files")
print(f"CẦN RESAMPLE (daily/other): {resample_count} files")



MONTHLY    (3 files):
  • cpi_mom.csv               |    360 rows | KEEP AS-IS
  • core_inflation_rate.csv   |    131 rows | KEEP AS-IS
  • m2.csv                    |    195 rows | KEEP AS-IS

ANNUAL     (1 files):
  • cpi_base_year.csv         |     30 rows | RESAMPLE

DAILY      (19 files):
  • interest_rate.csv         |    109 rows | RESAMPLE
  • brent.csv                 |  4,658 rows | RESAMPLE
  • wti.csv                   |  9,260 rows | RESAMPLE
  • gasoline.csv              |    279 rows | RESAMPLE
  • gasoline_world.csv        |  6,395 rows | RESAMPLE
  • natural_gas.csv           |  6,437 rows | RESAMPLE
  • gold.csv                  |  6,431 rows | RESAMPLE
  • silver.csv                |  6,433 rows | RESAMPLE
  • VNINDEX.csv               |  6,240 rows | RESAMPLE
  • VN30.csv                  |  4,290 rows | RESAMPLE
  • HNX.csv                   |  5,071 rows | RESAMPLE
  • UPCOM.csv                 |  4,177 rows | RESAMPLE
  • NASDAQ.csv                | 13,916 rows 

In [49]:
monthly_data = {}
agg_report = []

files_keep_monthly = {'cpi_mom.csv', 'core_inflation_rate.csv', 'm2.csv'}
files_downsample = set(freq_df[freq_df['Frequency'] == 'DAILY']['File'].values)  
files_annual = {'cpi_base_year.csv'}  # Annual - 1 value/year → repeat cho 12 tháng
files_quarterly = {'ppi_qoq.csv'}      # Quarterly - 1 value/quarter → repeat cho 3 tháng
files_irregular = {'interest_rate.csv'} # Irregular - dùng ffill

for filename, df in loaded_data.items():
    try:
        df_sorted = df.sort_values('date').copy()

        if filename in files_keep_monthly:
            monthly_data[filename] = df_sorted
            print(f" {filename:25} | KEEP MONTHLY | {len(df_sorted)} rows")
            agg_report.append({
                'Filename': filename,
                'Strategy': 'KEEP_MONTHLY',
                'Original_Rows': len(df),
                'Monthly_Rows': len(df_sorted),
                'Features': len(df_sorted.columns) - 1
            })

        elif filename in files_downsample:
            df_agg = df_sorted.copy()
            df_agg = df_agg.set_index('date')
            value_cols = [col for col in df_agg.columns]
            
            resampled = df_agg[value_cols].resample('MS').agg({
                col: ['last'] # lấy lâst trước 
                for col in value_cols
            })
            resampled.columns = ['_'.join(col).strip() for col in resampled.columns.values]
            resampled = resampled.reset_index()
            
            monthly_data[filename] = resampled
            compression_ratio = (len(resampled) / len(df)) * 100
            
            print(f"⬇ {filename:25} | DOWNSAMPLE Daily→Monthly | {len(df):6,} → {len(resampled):4,} rows | 5 features (last,mean,std,min,max)")
            agg_report.append({
                'Filename': filename,
                'Strategy': 'DOWNSAMPLE_DAILY',
                'Original_Rows': len(df),
                'Monthly_Rows': len(resampled),
                'Features': len(resampled.columns) - 1
            })

        elif filename in files_annual:
            df_agg = df_sorted.copy()
            df_agg = df_agg.set_index('date')
            value_cols = [col for col in df_agg.columns]
            

            resampled = df_agg[value_cols].resample('MS').ffill()
            resampled = resampled.reset_index()
            
            monthly_data[filename] = resampled
            print(f"⬆ {filename:25} | UPSAMPLE Annual→Monthly (repeat 12 months) | {len(df):6,} → {len(resampled):6,} rows | Forward Fill")
            agg_report.append({
                'Filename': filename,
                'Strategy': 'UPSAMPLE_ANNUAL_FFILL',
                'Original_Rows': len(df),
                'Monthly_Rows': len(resampled),
                'Features': len(resampled.columns) - 1
            })

        elif filename in files_quarterly:
            df_agg = df_sorted.copy()
            df_agg = df_agg.set_index('date')
            value_cols = [col for col in df_agg.columns]
  
            resampled = df_agg[value_cols].resample('MS').ffill()
            resampled = resampled.reset_index()
            
            monthly_data[filename] = resampled
            print(f"⬆ {filename:25} | UPSAMPLE Quarterly→Monthly (repeat 3 months) | {len(df):6,} → {len(resampled):6,} rows | Forward Fill")
            agg_report.append({
                'Filename': filename,
                'Strategy': 'UPSAMPLE_QUARTERLY_FFILL',
                'Original_Rows': len(df),
                'Monthly_Rows': len(resampled),
                'Features': len(resampled.columns) - 1
            })

        elif filename in files_irregular:
            df_agg = df_sorted.copy()
            df_agg = df_agg.set_index('date')
            value_cols = [col for col in df_agg.columns]
            
            resampled = df_agg[value_cols].resample('MS').ffill()
            resampled = resampled.reset_index()
            
            monthly_data[filename] = resampled
            print(f"⬆ {filename:25} | UPSAMPLE Irregular→Monthly | {len(df):6,} → {len(resampled):6,} rows | Forward Fill")
            agg_report.append({
                'Filename': filename,
                'Strategy': 'UPSAMPLE_IRREGULAR_FFILL',
                'Original_Rows': len(df),
                'Monthly_Rows': len(resampled),
                'Features': len(resampled.columns) - 1
            })
    
    except Exception as e:
        print(f"✗ {filename:25} | ERROR: {str(e)[:40]}")

print(f"\n Resample thành công: {len(monthly_data)} files")

agg_df = pd.DataFrame(agg_report)
for strategy in agg_df['Strategy'].unique():
    strategy_df = agg_df[agg_df['Strategy'] == strategy]
    total_features = strategy_df['Features'].sum()
    print(f"{strategy:30} | {len(strategy_df):2d} files | Total rows: {strategy_df['Original_Rows'].sum():,} → {strategy_df['Monthly_Rows'].sum():,} | Total features: {total_features:3d}")


 cpi_mom.csv               | KEEP MONTHLY | 360 rows
 core_inflation_rate.csv   | KEEP MONTHLY | 131 rows
⬆ cpi_base_year.csv         | UPSAMPLE Annual→Monthly (repeat 12 months) |     30 →    349 rows | Forward Fill
⬇ interest_rate.csv         | DOWNSAMPLE Daily→Monthly |    109 →  145 rows | 5 features (last,mean,std,min,max)
⬆ ppi_qoq.csv               | UPSAMPLE Quarterly→Monthly (repeat 3 months) |     40 →    118 rows | Forward Fill
 m2.csv                    | KEEP MONTHLY | 195 rows
⬇ brent.csv                 | DOWNSAMPLE Daily→Monthly |  4,658 →  226 rows | 5 features (last,mean,std,min,max)
⬇ wti.csv                   | DOWNSAMPLE Daily→Monthly |  9,260 →  436 rows | 5 features (last,mean,std,min,max)
⬇ gasoline.csv              | DOWNSAMPLE Daily→Monthly |    125 →  103 rows | 5 features (last,mean,std,min,max)
⬇ gasoline_world.csv        | DOWNSAMPLE Daily→Monthly |  2,534 →  324 rows | 5 features (last,mean,std,min,max)
⬇ natural_gas.csv           | DOWNSAMPLE Daily→Month

In [50]:
first_file = list(monthly_data.keys())[2]
sample_df = monthly_data[first_file]

print(f"\n📋 SAMPLE RESAMPLED FILE: {first_file}")
print(f"Shape: {sample_df.shape} (rows × columns)")
print(f"\nColumns: {list(sample_df.columns)}")
print(f"\nFirst 10 rows:")
print(sample_df.tail(30))

# Show statistics
print(f"\n📊 STATISTICS:")
print(sample_df.describe())


📋 SAMPLE RESAMPLED FILE: cpi_base_year.csv
Shape: (349, 2) (rows × columns)

Columns: ['date', 'cpi_base_year']

First 10 rows:
          date  cpi_base_year
319 2021-08-01         171.93
320 2021-09-01         171.93
321 2021-10-01         171.93
322 2021-11-01         171.93
323 2021-12-01         171.93
324 2022-01-01         177.35
325 2022-02-01         177.35
326 2022-03-01         177.35
327 2022-04-01         177.35
328 2022-05-01         177.35
329 2022-06-01         177.35
330 2022-07-01         177.35
331 2022-08-01         177.35
332 2022-09-01         177.35
333 2022-10-01         177.35
334 2022-11-01         177.35
335 2022-12-01         177.35
336 2023-01-01         183.12
337 2023-02-01         183.12
338 2023-03-01         183.12
339 2023-04-01         183.12
340 2023-05-01         183.12
341 2023-06-01         183.12
342 2023-07-01         183.12
343 2023-08-01         183.12
344 2023-09-01         183.12
345 2023-10-01         183.12
346 2023-11-01         183.12
3

In [51]:
# Merge tất cả vào trục thời gian chuẩn (Left Join)
merged_on_axis = time_index.copy()

for filename, df in monthly_data.items():
    cols_to_merge = [col for col in df.columns if col != 'date']
    if cols_to_merge:
        merge_df = df[['date'] + cols_to_merge].copy()
        
        rows_before = len(merged_on_axis)
        merged_on_axis = merged_on_axis.merge(merge_df, on='date', how='left')
        rows_after = len(merged_on_axis)
        
        print(f"  ✓ {filename:25} | {len(cols_to_merge):2d} features added | {rows_before} → {rows_after} rows")

print(f"Merged dataset shape: {merged_on_axis.shape}")
print(f"Date range: {merged_on_axis['date'].min().date()} to {merged_on_axis['date'].max().date()}")

  ✓ cpi_mom.csv               |  1 features added | 95 → 95 rows
  ✓ core_inflation_rate.csv   |  1 features added | 95 → 95 rows
  ✓ cpi_base_year.csv         |  1 features added | 95 → 95 rows
  ✓ interest_rate.csv         |  1 features added | 95 → 95 rows
  ✓ ppi_qoq.csv               |  1 features added | 95 → 95 rows
  ✓ m2.csv                    |  1 features added | 95 → 95 rows
  ✓ brent.csv                 |  1 features added | 95 → 95 rows
  ✓ wti.csv                   |  1 features added | 95 → 95 rows
  ✓ gasoline.csv              |  1 features added | 95 → 95 rows
  ✓ gasoline_world.csv        |  1 features added | 95 → 95 rows
  ✓ natural_gas.csv           |  1 features added | 95 → 95 rows
  ✓ gold.csv                  |  1 features added | 95 → 95 rows
  ✓ silver.csv                |  1 features added | 95 → 95 rows
  ✓ VNINDEX.csv               |  1 features added | 95 → 95 rows
  ✓ VN30.csv                  |  1 features added | 95 → 95 rows
  ✓ HNX.csv              

In [52]:
#Imputation với Forward Fill
missing_before = merged_on_axis.isnull().sum().sum()
print(f"  Missing values BEFORE: {missing_before:,}")

# Forward fill only (không back fill để không tạo giá trị tương lai)
merged_on_axis = merged_on_axis.fillna(method='ffill')

missing_after = merged_on_axis.isnull().sum().sum()
print(f"  Missing values AFTER ffill: {missing_after:,}")

# Show remaining nulls by column
null_cols = merged_on_axis.isnull().sum()
null_cols = null_cols[null_cols > 0].sort_values(ascending=False)

if len(null_cols) > 0:
    print("Remaining nulls by column (at start of time series):")
    for col, count in null_cols.items():
        null_pct = (count / len(merged_on_axis) * 100)
        print(f"{col:40} | {count:4d} nulls ({null_pct:.1f}%)")
else:
    print("\nNo remaining null values!")

print(f"Final dataset: {merged_on_axis.shape[0]} months × {merged_on_axis.shape[1]} columns")
print(f"Data completeness: {(1 - missing_after / (merged_on_axis.shape[0] * merged_on_axis.shape[1])) * 100:.1f}%")

  Missing values BEFORE: 149
  Missing values AFTER ffill: 0

No remaining null values!
Final dataset: 95 months × 25 columns
Data completeness: 100.0%


In [53]:
null_percentage = (merged_on_axis.isnull().sum() / len(merged_on_axis) * 100)
cols_to_drop = null_percentage[null_percentage > 90].index.tolist()

if len(cols_to_drop) > 0:
    print(f"Xóa {len(cols_to_drop)} cột có > 90% null:")
    for col in cols_to_drop:
        null_pct = null_percentage[col]
        print(f"  ✗ {col:40} | {null_pct:.1f}% null → DROP")
    
    merged_on_axis = merged_on_axis.drop(columns=cols_to_drop)
    print(f"\n✓ Sau khi xóa: {merged_on_axis.shape}")
else:
    print("✓ Không có cột nào cần xóa (tất cả ≤ 90% null)")

print(f"\n📊 CẬP NHẬT NULL STATS:")
missing_after_cleanup = merged_on_axis.isnull().sum().sum()
print(f"  Total nulls: {missing_after_cleanup:,}")
print(f"  Data completeness: {(1 - missing_after_cleanup / (merged_on_axis.shape[0] * merged_on_axis.shape[1])) * 100:.1f}%")


✓ Không có cột nào cần xóa (tất cả ≤ 90% null)

📊 CẬP NHẬT NULL STATS:
  Total nulls: 0
  Data completeness: 100.0%


In [54]:
# # Lấy đầy đủ tháng + last của tháng 
# monthly_data = {}
# agg_report = []

# for filename, df in loaded_data.items():
#     try:
#         df_agg = df.copy()
#         df_agg['year_month'] = df_agg['date'].dt.to_period('M')
        
#         # Get value columns
#         value_cols = [col for col in df_agg.columns if col not in ['date', 'year_month']]
        
#         # Aggregate (last value of month)
#         agg_dict = {col: 'last' for col in value_cols}
#         monthly = df_agg.groupby('year_month').agg(agg_dict).reset_index()
#         monthly['date'] = monthly['year_month'].dt.to_timestamp()
#         monthly = monthly.drop('year_month', axis=1)
        
#         monthly_data[filename] = monthly
        
#         compression_ratio = (len(monthly) / len(df)) * 100
#         agg_report.append({
#             'Filename': filename,
#             'Original': len(df),
#             'Monthly': len(monthly),
#             'Compression': f"{compression_ratio:.1f}%"
#         })
        
#         print(f"  ✓ {filename:25} | {len(df):6,} rows → {len(monthly):6,} months ({compression_ratio:.1f}% compression)")
    
#     except Exception as e:
#         print(f"  ✗ {filename:25} | ERROR: {str(e)[:40]}")

# print(f"\n✓ Gộp thành công: {len(monthly_data)} files")

# # Show summary
# agg_df = pd.DataFrame(agg_report)
# print(f"\nTổng rows ban đầu: {agg_df['Original'].sum():,}")
# print(f"Tổng tháng sau gộp: {agg_df['Monthly'].sum():,}")

## BƯỚC 4: Merge Tất Cả Dữ Liệu

In [55]:
# print("\n[STEP 3] MERGE TẤT CẢ DỮ LIỆU THEO NGÀY")
# print("="*100)

# # Check CPI data
# if 'cpi_mom.csv' not in monthly_data:
#     print("ERROR: CPI data not found!")
#     raise ValueError("CPI data (cpi_mom.csv) required as base")

# # Start with CPI as base
# cpi_df = monthly_data['cpi_mom.csv'].copy()
# value_col = [col for col in cpi_df.columns if col != 'date'][0]
# base = cpi_df[['date', value_col]].copy()
# base = base.rename(columns={value_col: 'CPI'})
# base = base.sort_values('date').reset_index(drop=True)

# print(f"  Base dataset (CPI): {len(base)} rows from {base['date'].min().date()} to {base['date'].max().date()}")

# # Merge other data
# merged_data = base.copy()
# merge_report = []

# for filename, df in monthly_data.items():
#     if filename != 'cpi_mom.csv':
#         cols_to_merge = [col for col in df.columns if col != 'date']
#         if cols_to_merge:
#             merge_df = df[['date'] + cols_to_merge].copy()
#             rows_before = len(merged_data)
#             merged_data = merged_data.merge(merge_df, on='date', how='outer')
#             rows_after = len(merged_data)
            
#             merge_report.append({
#                 'Source': filename,
#                 'New_Rows': rows_after - rows_before,
#                 'Total_Rows': rows_after,
#                 'Columns_Added': len(cols_to_merge)
#             })
            
#             print(f"  ✓ {filename:25} | +{rows_after - rows_before:4,} rows → {rows_after:6,} total")

# merged_data = merged_data.sort_values('date').reset_index(drop=True)

# print(f"\n✓ Merged dataset: {len(merged_data)} rows × {len(merged_data.columns)} columns")
# print(f"✓ Date range: {merged_data['date'].min().date()} to {merged_data['date'].max().date()}")

## BƯỚC 5: Xử Lý Missing Values

In [56]:
# print("\n[STEP 4] XỬ LÝ MISSING VALUES")
# print("="*100)

# # Filter to CPI data range
# cpi_available = merged_data[merged_data['CPI'].notna()].copy()
# cpi_start = cpi_available['date'].min()
# cpi_end = cpi_available['date'].max()

# print(f"  CPI data range: {cpi_start.date()} to {cpi_end.date()}")

# # Filter to CPI period
# merged_data = merged_data[(merged_data['date'] >= cpi_start) & 
#                           (merged_data['date'] <= cpi_end)].copy()

# print(f"  Filtered to CPI range: {len(merged_data)} rows")

# # Check missing values
# missing_before = merged_data.isnull().sum().sum()
# print(f"  Missing values BEFORE: {missing_before:,}")

# # Fill missing values
# merged_data = merged_data.fillna(method='ffill').fillna(method='bfill')

# missing_after = merged_data.isnull().sum().sum()
# print(f"  Missing values AFTER: {missing_after:,}")

# # Drop all NaN rows
# merged_data = merged_data.dropna(how='all', subset=merged_data.columns[1:])

# print(f"\n✓ Final rows after cleaning: {len(merged_data)}")
# print(f"✓ Data completeness: {(1 - missing_after / (merged_data.shape[0] * merged_data.shape[1])) * 100:.1f}%")

## BƯỚC 6: Feature Engineering

In [57]:
print("\n[STEP 5] FEATURE ENGINEERING")
print("="*100)

final_data = merged_on_axis.copy()

print(f"  Final dataset for modeling: {len(final_data)} months")
print(f"  Date range: {final_data['date'].min().date()} to {final_data['date'].max().date()}")

final_data['year'] = final_data['date'].dt.year
final_data['month'] = final_data['date'].dt.month
final_data['quarter'] = final_data['date'].dt.quarter

print(f"  ✓ Added temporal features (year, month, quarter)")
print(f"\n  Final shape: {final_data.shape}")
print(f"  Columns: {len(final_data.columns)}")


[STEP 5] FEATURE ENGINEERING
  Final dataset for modeling: 95 months
  Date range: 2018-06-01 to 2026-04-01
  ✓ Added temporal features (year, month, quarter)

  Final shape: (95, 28)
  Columns: 28


In [58]:
final_data.head()

,date,cpi_mom,core_inflation_rate,cpi_base_year,interest_rate_last,ppi_qoq,m2,brent_last,wti_last,gasoline_last,...,NASDAQ_last,S&P500_last,DAX_last,DOWJONES_last,NIKKEI225_last,HANGSENG_last,USDVND_last,year,month,quarter
0,2018-06-01,100.61,1.37,159.11,3.22,0.51,8879582.0,79.440002,72.46,18450.0,...,7510.299805,2718.370117,12306.000000,24271.410156,22304.509766,28955.109375,22956.0,2018,6,2
1,2018-07-01,99.91,1.41,159.11,3.22,1.08,8842552.0,74.250000,67.63,18450.0,...,7671.790039,2816.290039,12805.500000,25415.189453,22553.720703,28583.009766,23280.0,2018,7,3
2,2018-08-01,100.45,1.54,159.11,4.67,1.08,8888384.0,77.419998,69.37,18450.0,...,8109.540039,2901.520020,12364.059570,25964.820312,22865.150391,27888.550781,23303.0,2018,8,3
3,2018-09-01,100.59,1.61,159.11,4.67,1.08,8933435.0,82.720001,73.06,18450.0,...,8046.350098,2913.979980,12246.730469,26458.310547,24120.039062,27788.519531,23325.0,2018,9,3
4,2018-10-01,100.33,1.67,159.11,4.43,0.40,8999033.0,75.470001,65.44,18450.0,...,7305.899902,2711.739990,11447.509766,25115.759766,21920.460938,24979.689453,23343.0,2018,10,4


## BƯỚC 7: Kiểm Tra Chất Lượng Dữ Liệu

In [59]:
print("\n[STEP 6] KIỂM TRA CHẤT LƯỢNG DỮ LIỆU")
print("="*100)

print(f"  Final dataset shape: {final_data.shape}")
print(f"  Date range: {final_data['date'].min().date()} to {final_data['date'].max().date()}")
print(f"  Total columns: {len(final_data.columns)}")
print(f"  Numeric columns: {len(final_data.select_dtypes(include=[np.number]).columns)}")
print(f"  Missing values: {final_data.isnull().sum().sum()}")

print("\n📋 CHI TIẾT COLUMNS:")
print("-" * 100)
print(f"{'Column':<30} {'Type':<15} {'Missing':<10} {'Min':<15} {'Max':<15}")
print("-" * 100)

for col in final_data.columns[:15]:  # Show first 15 columns
    dtype = str(final_data[col].dtype)
    missing = final_data[col].isnull().sum()
    
    if final_data[col].dtype in ['float64', 'int64']:
        min_val = f"{final_data[col].min():.2f}"
        max_val = f"{final_data[col].max():.2f}"
    else:
        min_val = str(final_data[col].min())[:15]
        max_val = str(final_data[col].max())[:15]
    
    print(f"{col:<30} {dtype:<15} {missing:<10} {min_val:<15} {max_val:<15}")


[STEP 6] KIỂM TRA CHẤT LƯỢNG DỮ LIỆU
  Final dataset shape: (95, 28)
  Date range: 2018-06-01 to 2026-04-01
  Total columns: 28
  Numeric columns: 27
  Missing values: 0

📋 CHI TIẾT COLUMNS:
----------------------------------------------------------------------------------------------------
Column                         Type            Missing    Min             Max            
----------------------------------------------------------------------------------------------------
date                           datetime64[ns]  0          2018-06-01 00:0 2026-04-01 00:0
cpi_mom                        float64         0          98.46           101.52         
core_inflation_rate            float64         0          0.49            5.21           
cpi_base_year                  float64         0          159.11          189.76         
interest_rate_last             float64         0          1.20            9.43           
ppi_qoq                        float64         0          -1.66   

## BƯỚC 8: Lưu Unified Dataset

In [60]:
print("\n[STEP 7] SAVING UNIFIED DATASET")
print("="*100)

# Save only the main unified dataset
output_file = os.path.join(data_processed_dir, 'cpi_forecast_full_dataset_advanced.csv')
final_data.to_csv(output_file, index=False, encoding='utf-8')
file_size = os.path.getsize(output_file) / 1024  # KB

print(f"  ✓ Saved: cpi_forecast_full_dataset_advanced.csv ({len(final_data)} rows, {file_size:.1f} KB)")
print(f"  📁 Location: {output_file}")


[STEP 7] SAVING UNIFIED DATASET
  ✓ Saved: cpi_forecast_full_dataset_advanced.csv (95 rows, 26.6 KB)
  📁 Location: d:\Nam4-HK1\KLTN\Vietnam_economic_lakehouse\model_test\data\processed\cpi_forecast_full_dataset_advanced.csv


## BƯỚC 9: Tổng Kết & Thống Kê

In [61]:
print("\n" + "="*120)
print("UNIFIED DATASET - TỔNG KẾT")
print("="*120)

summary_text = f"""
📊 THỐNG KÊ DỮ LIỆU:
  • Total data sources loaded: {len(loaded_data)} files (out of {len(data_config)} defined)
  • Date range: {final_data['date'].min().date()} to {final_data['date'].max().date()}
  • Monthly observations: {len(final_data)}
  • Total features: {len(final_data.columns) - 1} (excluding date)
  • Data completeness: 100%
  • Missing values: {final_data.isnull().sum().sum()}

📂 FEATURE CATEGORIES:
  • Economic indicators: 6 sources (CPI, Core Inflation, Interest Rates, M2, PPI, Base Year)
  • Commodities: 7 sources (Brent, WTI, Gasoline, Gold, Silver, Natural Gas)
  • Vietnam stocks: 4 indices (VNINDEX, VN30, HNX, UPCOM)
  • Global stocks: 5 indices (NASDAQ, S&P500, DAX, Nikkei225, Hang Seng)
  • Currency: 1 (USDVND)
  • Temporal features: 3 (year, month, quarter)

✅ OUTPUTS GENERATED:
  ✓ cpi_forecast_full_dataset_advanced.csv
    - {len(final_data)} rows × {len(final_data.columns)} columns
    - Date range: {final_data['date'].min().date()} to {final_data['date'].max().date()}
    - Ready for STEP 1: Data Preparation & Exploration

🚀 NEXT STEPS:
  1. Run: step1_data_preparation.ipynb
     - Time series decomposition
     - Structural breaks detection
     - Statistical transformations
     - Multicollinearity analysis
  
  2. Run: step1_visualizations_thesis.ipynb
     - Generate thesis tables & figures
     - Create publication-ready visualizations (300 DPI)
"""

print(summary_text)

# Show data summary
print("\n📈 DATA SAMPLE (first 5 rows):")
print("-" * 150)
print(final_data.head().to_string())


UNIFIED DATASET - TỔNG KẾT

📊 THỐNG KÊ DỮ LIỆU:
  • Total data sources loaded: 24 files (out of 24 defined)
  • Date range: 2018-06-01 to 2026-04-01
  • Monthly observations: 95
  • Total features: 27 (excluding date)
  • Data completeness: 100%
  • Missing values: 0

📂 FEATURE CATEGORIES:
  • Economic indicators: 6 sources (CPI, Core Inflation, Interest Rates, M2, PPI, Base Year)
  • Commodities: 7 sources (Brent, WTI, Gasoline, Gold, Silver, Natural Gas)
  • Vietnam stocks: 4 indices (VNINDEX, VN30, HNX, UPCOM)
  • Global stocks: 5 indices (NASDAQ, S&P500, DAX, Nikkei225, Hang Seng)
  • Currency: 1 (USDVND)
  • Temporal features: 3 (year, month, quarter)

✅ OUTPUTS GENERATED:
  ✓ cpi_forecast_full_dataset_advanced.csv
    - 95 rows × 28 columns
    - Date range: 2018-06-01 to 2026-04-01
    - Ready for STEP 1: Data Preparation & Exploration

🚀 NEXT STEPS:
  1. Run: step1_data_preparation.ipynb
     - Time series decomposition
     - Structural breaks detection
     - Statistical tra

## Kiểm Tra Lại Kết Quả

In [62]:
# Verify saved file
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
    verify_df = pd.read_csv(output_file, nrows=5)
    
    print("✅ VERIFICATION SUCCESSFUL")
    print(f"  ✓ File exists: {output_file}")
    print(f"  ✓ File size: {file_size:.2f} MB")
    print(f"  ✓ Rows: {len(pd.read_csv(output_file))}")
    print(f"  ✓ Columns: {len(verify_df.columns)}")
    print(f"  ✓ Date range: {verify_df['date'].iloc[0]} to {verify_df['date'].iloc[-1]}")
    print("\n✓ Ready for STEP 1: Data Preparation & Exploration")
else:
    print("❌ ERROR: File not saved")

✅ VERIFICATION SUCCESSFUL
  ✓ File exists: d:\Nam4-HK1\KLTN\Vietnam_economic_lakehouse\model_test\data\processed\cpi_forecast_full_dataset_advanced.csv
  ✓ File size: 0.03 MB
  ✓ Rows: 95
  ✓ Columns: 28
  ✓ Date range: 2018-06-01 to 2018-10-01

✓ Ready for STEP 1: Data Preparation & Exploration
